# Post 1: a small ResNet on Imagenette, on a Colab TPU

CIFAR-10's 32x32 photos are hard to look inside: by the last layers there are only a
few pixels left to see. Imagenette is ten easy classes of ImageNet at 160x160, about 5
times the resolution, so feature maps stay big enough to show shapes.

Checkpoints are written straight to a GCS bucket and metrics go to Weights & Biases,
so both outlive the runtime and can be read from anywhere. **If the runtime goes
away, start a new one and run the notebook again: the run carries on from its newest
checkpoint in the bucket, and a finished run is skipped.**

`COLAB.md` explains how to run this on a Colab TPU with `colab exec`, which runs every
cell in order with nobody at the keyboard.

## Setup

On Colab this clones the branch below and installs it. `BRANCH` must already be pushed,
because the runtime sees GitHub, not your working copy. Locally it does nothing.

The repo needs a JAX recent enough to have `jax.set_mesh`. If the runtime's is older,
this upgrades it before anything imports JAX.

In [ ]:
import os
import pathlib
import sys

BRANCH = "p01-data-pipeline"

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or "google.colab" in sys.modules
if IN_COLAB:
    if not pathlib.Path("/content/toylib").exists():
        !git clone --depth 1 --branch {BRANCH} https://github.com/anujkhare/toylib.git /content/toylib
    %cd /content/toylib
    !git pull --ff-only origin {BRANCH}
    !pip install -q -e ".[harness,viz,gcs]"
    has_set_mesh = !python -c "import jax; print(hasattr(jax, 'set_mesh'))"
    if has_set_mesh[-1] != "True":
        !pip install -q -U "jax[tpu]"

## Sign in to GCS and Weights & Biases

`colab exec` can't answer prompts, so this cell asks for nothing. `COLAB.md` shows how a
service-account key and the W&B key reach the runtime as files. The key isn't passed with `colab exec --env`, because that saves it in plain
text in the CLI's local history.

If the runtime can't read the bucket, this stops here rather than failing hours later.

In [ ]:
WANDB_PROJECT = "toylib-p01-imagenette"
WANDB_ENTITY = "anujkhare18"
BUCKET = "gs://tinystories-checkpoints"
CHECKPOINT_ROOT = f"{BUCKET}/toylib-p01-imagenette"
WANDB_KEY_FILE = pathlib.Path("/content/.wandb_key")

if IN_COLAB:
    project = os.environ.get("GOOGLE_CLOUD_PROJECT")
    if project:
        !gcloud config set project {project}
    bucket_access = !gcloud storage ls {BUCKET}/ 2>&1 | head -1
    if not bucket_access or not bucket_access[0].startswith("gs://"):
        raise RuntimeError(f"Can't read the bucket, so check the service-account key; see COLAB.md: {bucket_access}")
    if "WANDB_API_KEY" not in os.environ:
        if not WANDB_KEY_FILE.exists():
            raise RuntimeError(f"Upload the W&B key to {WANDB_KEY_FILE} first; see COLAB.md.")
        os.environ["WANDB_API_KEY"] = WANDB_KEY_FILE.read_text().strip()

## The run

| Setting | Wide ResNet-20 on CIFAR-10 | This run |
| --- | --- | --- |
| Images | 50,000 at 32x32 | 9,469 at 160x160 |
| Stem | 3x3 | 7x7, stride 2 |
| Stages | 3, 3 blocks each, up to 128 channels | 4, 2 blocks each, up to 256 channels |
| Parameters | 1.08M | 2.80M |
| Epochs | 100 | 200 |
| Augmentation | Flip, crop, 16x16 cutout | Flip, random resized crop, 40x40 cutout |
| Optimizer | AdamW, 1e-3, cosine decay, weight decay 0.1 | Same |

This is the layout of ResNet-18 at half its width. The stem and the extra stage take the
image from 160x160 down to 10x10 before global average pooling. A 7x7 stem also has
filters big enough to look at. There is a fifth of CIFAR-10's data, each step is bigger,
and epochs are shorter, so the run has twice as many of them.

Random resized crops zoom into a part of each photo, which is the main defence against
overfitting so few images.

The harness drops a partial batch, so evaluation during training covers 3,600 of the
3,925 test images, picked at random. The final accuracy below uses all of them.

In [ ]:
import jax
import numpy as np
import optax
from absl import logging as absl_logging

from toylib.harness import analyze, inference
from toylib.harness.experiment import (
    CheckpointConfig,
    EvalConfig,
    Experiment,
    LoggerConfig,
    MultiOptimizerConfig,
    OptimizerConfig,
    Task,
    TrainingConfig,
    WandBLoggerConfig,
)
from toylib.harness.metrics import Loss
from toylib_projects.p01_cnn_mnist import data, metrics, model, resnet

RUNS = {
    "resnet18w32": resnet.ResNetConfig(
        image_shape=(160, 160, 3),
        width=32,
        blocks_per_stage=2,
        num_stages=4,
        stem_kernel_size=7,
        stem_stride=2,
    )
}
EPOCHS = 200
BATCH_SIZE = 128
EVAL_BATCH_SIZE = 400
CUTOUT_SIZE = 40
WEIGHT_DECAY = 0.1
CHECKPOINT_EVERY_EPOCHS = 10  # The most work a lost runtime can cost.

absl_logging.set_verbosity(absl_logging.ERROR)  # quiet orbax's read-only notices
print("JAX", jax.__version__, "| devices:", jax.devices())
if IN_COLAB and jax.default_backend() == "cpu":
    raise RuntimeError("No accelerator on this runtime, so training would take days.")

In [ ]:
def make_experiment(name, logger_config=None, checkpoint_dir=None):
    """One run through the harness. Defaults to the full run, on W&B and GCS."""
    train_ds, test_ds = data.make_imagenette_datasets(
        batch_size=BATCH_SIZE, eval_batch_size=EVAL_BATCH_SIZE, cutout_size=CUTOUT_SIZE
    )
    steps_per_epoch = train_ds.batches_per_epoch
    total_steps = EPOCHS * steps_per_epoch
    run_id = f"{name}_e{EPOCHS}"
    out_dir = pathlib.Path(model.__file__).parent / "outputs" / f"imagenette_{run_id}"

    learning_rate = optax.cosine_decay_schedule(1e-3, total_steps)
    optimizer_config = MultiOptimizerConfig(
        [OptimizerConfig("adamw", optax.adamw(learning_rate, weight_decay=WEIGHT_DECAY))],
        optimizer_for_param=lambda path: "adamw",
    )
    task_metrics = [Loss(), metrics.Accuracy()]
    return Experiment(
        train_task=Task("train", train_ds, metrics=task_metrics),
        eval_task=Task("test", test_ds, metrics=task_metrics),
        model_factory=resnet.build_resnet,
        forward_fn=model.forward,
        model_config=RUNS[name],
        # +1: the harness evaluates and saves a step before counting it, so this
        # makes the last epoch's evaluation and checkpoint happen.
        training_config=TrainingConfig(
            optimizer_config=optimizer_config, max_steps=total_steps + 1
        ),
        eval_config=EvalConfig(
            eval_interval_steps=steps_per_epoch, num_eval_steps=test_ds.batches_per_epoch
        ),
        checkpoint_config=CheckpointConfig(
            save_interval_steps=CHECKPOINT_EVERY_EPOCHS * steps_per_epoch,
            max_to_keep=None,  # Tens of MB each, and useful for later figures.
            checkpoint_dir=checkpoint_dir or f"{CHECKPOINT_ROOT}/{run_id}",
            checkpoint_dataset_iterator=True,  # Resume mid-epoch with the same order.
        ),
        logger_config=logger_config
        or WandBLoggerConfig(
            project_name=WANDB_PROJECT,
            user_name=WANDB_ENTITY,
            log_dir=str(out_dir),
            run_id=run_id,  # A repeat of this run resumes the same W&B run.
            train_log_interval_steps=50,
            also_log_to_file=True,
        ),
    )

## How long will this take?

A short timing run, saved nowhere: 50 training steps and one evaluation
after compiling. It also proves that the model, sharding and evaluation work on this
accelerator before hours go into them. A step here is the same size as a real step, so
the estimate covers the whole run, including one evaluation per epoch. The first save to
the bucket happens at the very start of training, so a permissions problem shows up in
the next cell straight away.

In [ ]:
import tempfile
import time

TIMING_STEPS = 50  # Fewer than an epoch, which is 73 steps.
estimates = {}
for name, config in RUNS.items():
    with tempfile.TemporaryDirectory() as tmp:
        tmp = pathlib.Path(tmp)
        exp = make_experiment(
            name,
            logger_config=LoggerConfig(log_dir=str(tmp), run_id="timing"),
            checkpoint_dir=str(tmp / "checkpoints"),
        )
        exp.init_state()
        batches = iter(exp.train_task.dataset)
        exp.inner_loop(next(batches))  # Compiles the training step.
        jax.block_until_ready(exp.model)

        start = time.monotonic()
        for _ in range(TIMING_STEPS):
            exp.inner_loop(next(batches))
        jax.block_until_ready(exp.model)
        seconds_per_step = (time.monotonic() - start) / TIMING_STEPS

        exp.run_validation()  # Compiles the evaluation step.
        start = time.monotonic()
        exp.run_validation()
        seconds_per_eval = time.monotonic() - start

        n_params = analyze.print_param_sizes(exp.model, depth=1)[1]
        steps = exp.training_config.max_steps
        hours = (steps * seconds_per_step + EPOCHS * seconds_per_eval) / 3600
        exp.cleanup()
    estimates[name] = hours
    print(f"{name:12s} {n_params:>10,} params  {1 / seconds_per_step:6.1f} steps/s  "
          f"{seconds_per_eval:5.1f} s/eval  about {hours:.2f} h for {EPOCHS} epochs")
print(f"total: about {sum(estimates.values()):.2f} h")

## Train

Prints a W&B link, where the curves appear as it trains. It starts from the newest
checkpoint in the bucket if there is one.

In [ ]:
results = {}
for name in RUNS:
    exp = make_experiment(name)
    try:
        exp.init_state()
        resumed = exp.resume()
        print(f"\n{name}: " + (f"resuming at step {exp.step}" if resumed else "starting fresh")
              + f" of {exp.training_config.max_steps}")
        exp.outer_loop()
        results[name] = exp.run_validation()
        images = data.center_crop(exp.eval_task.dataset.arrays["image"], 160)
        logits = inference.batched_apply(
            lambda m, b: m(b["image"]),
            exp.model,
            {"image": data.preprocess(images, data.DATASETS["imagenette"])},
            EVAL_BATCH_SIZE,
        )
        labels = exp.eval_task.dataset.arrays["label"]
        results[name]["full_test_accuracy"] = float((logits.argmax(-1) == labels).mean())
    finally:
        exp.cleanup()  # Waits for W&B to finish syncing.
    print(f"{name}: test accuracy {results[name]['full_test_accuracy']:.4f}, "
          f"test loss {results[name]['val/loss']:.4f}")

## Results

For reference, the fast.ai leaderboard reaches 91.6% in 20 epochs and 94.2% in 200 at
128x128, with larger models. Checkpoints stay in the bucket under `CHECKPOINT_ROOT`, one
every 10 epochs.

In [ ]:
for name, result in results.items():
    print(f"{name:12s} test accuracy {result['full_test_accuracy']:.4f}  test loss {result['val/loss']:.4f}")